# Solver shootout: every method on one problem

This notebook accompanies the docs page [`solver-shootout`](../../docs/examples/solver-shootout.md). One two-parameter problem is handed to every solver the library dispatches and to the three canonical baselines, and all of them are scored the same way on the same held-out split. The docs page tells the story at a small sample size; this notebook runs the full study, prints both tables, and reproduces the budget sweep.

Set `SCOREQUANT_EXAMPLE_FAST=1` to shrink every sample and optimizer budget for a quick pass.

## The problem and its geometry

`two_parameter_gaussian_mixture` places two overlapping isotropic Gaussian bumps on a bounded square. Each event's weight is the mixture intensity and each event's score is the exact linear component score. Two components means two parameters and two score columns.

One structural fact drives every result below. An exact linear component score satisfies $\sum_k c_k s_k = 1$ identically, so with two components the whole score cloud lies on a single affine line in the plane. The Fisher information is still full rank — scores are never centered — but the variation in score space is one-dimensional.

Double precision is an application-level choice. The library never sets it at import time, so the notebook turns it on itself, before anything computes.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.solver_shootout import (
    HEADLINE_BINS,
    SCALE_FACTORS,
    SCALE_PROBE_BINS,
    compare_methods,
    projection_direction,
    rectangular_gap_sweep,
    retention,
    scale_sensitivity,
    whitening_probe,
)
from examples.synthetic_problems import two_parameter_gaussian_mixture

sizes = example_scale((4_000, 2_000, 15_000), (800, 400, 2_000))
problem = two_parameter_gaussian_mixture(n_bins=HEADLINE_BINS, sizes=sizes)
train, test = problem.train, problem.test

{
    "training events": int(train.scores.shape[0]),
    "held-out events": int(test.scores.shape[0]),
    "bin budget": problem.n_bins,
    "score columns": int(train.scores.shape[1]),
    "scores lie on one line": bool(np.allclose(train.scores.sum(axis=1), 2.0)),
}

## One solver, explicitly

Before running the whole table, here is a single fit written out in full: a `ScoreSample` source, the D-optimality criterion, and the exact positive-gain exchange solver. The returned rule predicts labels for any score rows, so the held-out number is a genuine out-of-sample measurement rather than a refit.

In [ ]:
source = sq.ScoreSample(train.scores, train.weights)
rule = sq.fit_quantizer(
    source,
    n_bins=problem.n_bins,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=7),
)

{
    "train D-efficiency": retention(
        train.scores,
        np.asarray(rule.predict_scores(train.scores)),
        train.weights,
        problem.n_bins,
    ),
    "held-out D-efficiency": retention(
        test.scores,
        np.asarray(rule.predict_scores(test.scores)),
        test.weights,
        problem.n_bins,
    ),
    "informative rank": rule.rank,
    "source kind": rule.source_kind,
}

## Every applicable solver and the three baselines

`compare_methods` walks the dispatch table: both partition solvers, all five score-space quantizer solvers, then the rectangular observation grid, Euclidean k-means on raw scores, and equal-frequency bins on the stated one-dimensional projection. Every labeling — including every baseline's — is scored with `information_report`.

Costs are wall-clock medians measured here, on this machine, after one warm-up call that absorbs tracing and compilation. Absolute seconds are hardware-specific, so the last column reports the ratio to the fastest information-aware fit.

In [ ]:
methods = compare_methods(
    problem,
    soft_steps=example_scale(400, 60),
    timing_repeats=example_scale(5, 1),
)
fastest = min(entry.seconds for entry in methods if entry.family == "information_aware")

header = f"{'method':<34}{'task':<20}{'train':>10}{'held-out':>11}{'cost':>8}"
print(header)
print("-" * len(header))
for entry in methods:
    held_out = "-" if entry.test_retention is None else f"{entry.test_retention:.5f}"
    print(
        f"{entry.label:<34}{entry.task:<20}{entry.train_retention:>10.5f}"
        f"{held_out:>11}{entry.seconds / fastest:>8.2f}"
    )

A partition result carries no held-out number because `optimize_partition` deliberately returns no predictor: it labels one fixed sample and claims nothing about any other. The `fit_quantizer` row directly beneath it is the same solver compiled into a reusable rule.

The information-aware solvers agree with each other to about one part in a hundred thousand. The rectangular observation grid does not.

## Score space against observation space, across bin budgets

A single bin budget could be lucky. The sweep below repeats the comparison at perfect-square budgets, where the equal-width grid always receives an exact cell count.

In [ ]:
sweep = rectangular_gap_sweep((4, 9, 16, 25), sizes=sizes)
for row in sweep:
    print(
        f"{int(row['n_bins']):>3} bins   score space {row['score_space']:.5f}   "
        f"grid {row['observation_space']:.5f}   gap {row['gap']:.5f}"
    )

fig, ax = plt.subplots(figsize=(6.5, 4))
budgets = [row["n_bins"] for row in sweep]
ax.plot(budgets, [row["score_space"] for row in sweep], marker="o", label="score space")
ax.plot(
    budgets,
    [row["observation_space"] for row in sweep],
    marker="s",
    label="rectangular observation grid",
)
ax.set(
    xlabel="bin budget",
    ylabel="held-out D-efficiency",
    xticks=budgets,
    title="The gap holds at every bin budget",
)
ax.legend();

The grid is not merely worse, it is erratically worse: its nine-bin value falls below its four-bin value, because a three-by-three grid straddles the mixture's symmetry axis and produces cells whose score means nearly coincide. Adding cells to an observation-space grid does not reliably add information.

## What whitening buys

On this problem the whitened and unwhitened fits agree almost exactly, and that is a fact about the geometry rather than about whitening. The score cloud lies on a line, every linear map acts along a line as one uniform rescaling, and k-means is invariant to a uniform rescaling. The metric has nowhere to act.

In [ ]:
whitening = whitening_probe(problem)
print(
    f"whitened {whitening['whitened']:.9f}   unwhitened {whitening['unwhitened']:.9f}   "
    f"difference {abs(whitening['whitened'] - whitening['unwhitened']):.2e}"
)

direction = projection_direction(problem)
print("one-dimensional projection direction:", np.round(direction, 6))

To see whitening do something, the score cloud needs more than one direction of variation. Three components give one. The probe below multiplies one score column of a signal-plus-two-backgrounds problem by a constant and refits. Multiplying a score column is a reparameterization — a change of units for one coefficient — and D-efficiency is invariant under it, so any movement is a method reacting to units it should not be able to see.

In [ ]:
probe = scale_sensitivity(scales=SCALE_FACTORS, n_bins=SCALE_PROBE_BINS, sizes=sizes)
for row in probe:
    print(
        f"multiplier {row['scale']:>6.1f}   whitened {row['whitened']:.9f}   "
        f"Euclidean {row['euclidean']:.5f}"
    )

## Interpretation

Three things separate cleanly on this problem.

Binning in score space beats binning the raw variables, by roughly six to thirty D-efficiency points depending on the bin budget, and the observation-space grid behaves non-monotonically in that budget.

Whitening buys invariance to the units of the score columns. The shootout's own two-parameter problem cannot show that, because its score cloud is one-dimensional; the three-parameter probe shows it starkly, with Euclidean k-means collapsing under a change of units that the whitened fit does not even notice.

Among the information-aware solvers the retention differences are far smaller than the cost differences. That is not evidence that these solvers are interchangeable — it is evidence that this problem does not separate them. The counterexample pages show problems that do.

The committed four-panel figure on the docs page is regenerated by running `python -m examples.solver_shootout`, which writes both the figure and the machine-readable metrics the published tables are checked against.